# Get To Know A Dataset: NASA PDS Geosciences Node Planetary Science Archive

This notebook introduces the [**NASA Planetary Data System (PDS) Geosciences Node**](https://pds-geosciences.wustl.edu/) planetary science archive on the [Registry of Open Data on AWS](https://registry.opendata.aws/). The archive holds calibrated and derived data for the surfaces and interiors of terrestrial planetary bodies—including Mars, the Moon, Venus, and Mercury—from missions such as Mars Reconnaissance Orbiter, Mars Science Laboratory (Curiosity), Mars 2020 (Perseverance), Mars Odyssey, Mars Express, Lunar Reconnaissance Orbiter, and others.

| Resource | Value |
|----------|-------|
| **S3 bucket** | `pds-geosciences` |
| **Region** | `us-west-2` |
| **Access** | Public, anonymous (unsigned requests) |
| **Documentation** | [Geosciences Node](https://pds-geosciences.wustl.edu/) |

> **Note (draft onboarding):** The bucket may not exist or may be empty while AWS Open Data onboarding completes. All listing and loading steps below are written to **fail gracefully** with an explanatory message rather than raising uncaught errors.


## Dataset organization (expected layout)

PDS archives are organized as **standards-compliant products** rather than ad hoc files. On AWS, objects are expected to mirror the Geosciences Node archive layout under S3 prefixes (for example by mission, instrument, or data release). Typical product types include:

- **PDS3** label + image pairs (`.lbl` / `.img`, or detached labels)
- **PDS4** bundles with XML labels and binary or image extensions
- **Cloud-friendly rasters** where available (for example Cloud Optimized GeoTIFF)

Exact prefix names will appear once data are published to `s3://pds-geosciences/`. The code in the next sections discovers whatever structure is present at run time.


In [ ]:
# Library requirements are listed in pds-geosciences/requirements.txt
# Install with: pip install -r pds-geosciences/requirements.txt

from __future__ import annotations

import io
from typing import Any

import boto3
import matplotlib.pyplot as plt
import numpy as np
import s3fs
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

BUCKET = "pds-geosciences"
REGION = "us-west-2"

# Unsigned S3 client for public bucket access (AWS Open Data pattern)
s3 = boto3.client(
    "s3",
    region_name=REGION,
    config=Config(signature_version=UNSIGNED),
)

# s3fs provides a filesystem-like interface to the same bucket
fs = s3fs.S3FileSystem(anon=True, client_kwargs={"region_name": REGION})

print(f"Ready to access s3://{BUCKET}/ in {REGION} (anonymous)")


## Browse the bucket with boto3 and s3fs

We list top-level prefixes with `list_objects_v2` (boto3) and compare with `s3fs` globbing. If the bucket is missing or empty, you will see a short status message instead of a traceback.


In [ ]:
def bucket_status(s3_client, bucket: str) -> dict[str, Any]:
    """Return {'ok': bool, 'message': str, 'prefixes': list[str]}."""
    try:
        s3_client.head_bucket(Bucket=bucket)
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code", "")
        if code in ("404", "NoSuchBucket", "NotFound"):
            return {
                "ok": False,
                "message": (
                    f"Bucket s3://{bucket}/ does not exist yet (or is not public in {REGION}). "
                    "This is expected during AWS Open Data onboarding; re-run this notebook after "
                    "the registry entry and bucket go live."
                ),
                "prefixes": [],
            }
        return {"ok": False, "message": f"head_bucket failed: {code} — {exc}", "prefixes": []}

    resp = s3_client.list_objects_v2(Bucket=bucket, Delimiter="/", MaxKeys=1000)
    prefixes = [p["Prefix"] for p in resp.get("CommonPrefixes", [])]
    if not prefixes and resp.get("KeyCount", 0) == 0:
        return {
            "ok": True,
            "message": f"Bucket s3://{bucket}/ exists but has no listable top-level prefixes yet (may be empty).",
            "prefixes": [],
        }
    return {
        "ok": True,
        "message": f"Found {len(prefixes)} top-level prefix(es) in s3://{bucket}/.",
        "prefixes": prefixes,
    }


status = bucket_status(s3, BUCKET)
print(status["message"])
if status["prefixes"]:
    print("\nTop-level prefixes (boto3):")
    for prefix in status["prefixes"][:20]:
        print(" ", prefix)
    if len(status["prefixes"]) > 20:
        print(f"  ... and {len(status['prefixes']) - 20} more")


In [ ]:
if status["ok"]:
    s3_path = f"s3://{BUCKET}/"
    try:
        # s3fs: list immediate children (non-recursive)
        entries = fs.ls(s3_path, detail=False)
        print(f"s3fs ls({s3_path}) returned {len(entries)} entries (showing up to 15):")
        for entry in entries[:15]:
            print(" ", entry)
    except FileNotFoundError:
        print("s3fs could not list the bucket root (bucket missing or no access).")
    except Exception as exc:  # noqa: BLE001 — tutorial notebook; surface any s3fs error clearly
        print(f"s3fs listing skipped: {type(exc).__name__}: {exc}")
else:
    print("Skipping s3fs browse because bucket_status reported the bucket is unavailable.")


In [ ]:
MISSION_HINTS = ("mars", "mro", "msl", "m2020", "lro", "moon", "lunar", "venus", "mercury", "odyssey", "mex")
PLOT_EXTENSIONS = (".tif", ".tiff", ".jpg", ".jpeg", ".png", ".img")


def find_candidate_keys(s3_client, bucket: str, max_keys: int = 500) -> list[str]:
    """Scan up to max_keys objects and return paths that look like plottable planetary products."""
    if not status["ok"]:
        return []
    keys: list[str] = []
    paginator = s3_client.get_paginator("list_objects_v2")
    try:
        for page in paginator.paginate(Bucket=bucket, PaginationConfig={"MaxItems": max_keys}):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                lower = key.lower()
                if not lower.endswith(PLOT_EXTENSIONS):
                    continue
                if any(hint in lower for hint in MISSION_HINTS):
                    keys.append(key)
                elif len(keys) < 5:  # keep a few unlabeled examples if no mission match yet
                    keys.append(key)
    except ClientError as exc:
        print(f"Object listing failed: {exc}")
        return []
    return keys


candidates = find_candidate_keys(s3, BUCKET)
print(f"Candidate product keys for demo plotting: {len(candidates)}")
for key in candidates[:10]:
    print(" ", key)


## Open representative products and plot

When data are available, we prefer products whose keys suggest **Mars** or **Moon/LRO** origin, and formats that raster libraries can read directly (GeoTIFF/COG, PNG, JPEG). Raw PDS3 `.IMG` files may require mission-specific metadata; this notebook attempts a simple load when possible and otherwise reports what to try next (for example GDAL with the PDS driver or [`planetaryimage`](https://github.com/planetarypy/planetaryimage)).


In [ ]:
def load_image_array_from_s3(s3_client, bucket: str, key: str) -> tuple[np.ndarray | None, str]:
    """Load a 2-D array for plotting. Returns (array, message)."""
    lower = key.lower()
    try:
        body = s3_client.get_object(Bucket=bucket, Key=key)["Body"].read()
    except ClientError as exc:
        return None, f"get_object failed: {exc}"

    if lower.endswith((".tif", ".tiff")):
        try:
            import rasterio
            with rasterio.open(io.BytesIO(body)) as dataset:
                data = dataset.read(1)
            return data, "Loaded band 1 with rasterio"
        except Exception as exc:  # noqa: BLE001
            return None, f"rasterio could not read {key}: {exc}"

    if lower.endswith((".png", ".jpg", ".jpeg")):
        try:
            from PIL import Image
            image = Image.open(io.BytesIO(body))
            return np.asarray(image), f"Loaded with Pillow ({image.mode})"
        except Exception as exc:  # noqa: BLE001
            return None, f"Pillow could not read {key}: {exc}"

    if lower.endswith(".img"):
        # Minimal fallback: square dimension guess from file size (often wrong for PDS IMG)
        side = int(np.sqrt(len(body)))
        if side > 0 and side * side == len(body):
            arr = np.frombuffer(body, dtype=np.uint8).reshape((side, side))
            return arr, f"Interpreted .IMG as {side}x{side} uint8 (verify with PDS label)"
        return None, (
            "PDS3 .IMG usually needs its .LBL label (dimensions, dtype, band storage). "
            "Download the label pair from the same directory or use GDAL PDS driver."
        )

    return None, f"No loader implemented for extension of {key}"


def plot_product(s3_client, bucket: str, key: str) -> None:
    data, msg = load_image_array_from_s3(s3_client, bucket, key)
    print(msg)
    if data is None:
        return
    if data.ndim == 3:
        data = data[..., 0]
    finite = np.isfinite(data) if np.issubdtype(data.dtype, np.floating) else np.ones_like(data, dtype=bool)
    if finite.any():
        vmin, vmax = np.percentile(data[finite], [2, 98])
    else:
        vmin, vmax = None, None
    plt.figure(figsize=(8, 8))
    plt.imshow(data, cmap="gray", vmin=vmin, vmax=vmax)
    plt.title(key.split("/")[-1][:80])
    plt.axis("off")
    plt.tight_layout()
    plt.show()


if not status["ok"]:
    print("Bucket unavailable — skipping product plots until s3://pds-geosciences/ is published.")
elif not candidates:
    print(
        "No plottable candidate objects found in the first scan. "
        "After data upload, re-run find_candidate_keys() or set SAMPLE_KEYS manually."
    )
    SAMPLE_KEYS: list[str] = []
else:
    SAMPLE_KEYS = candidates[:2]
    print("Plotting up to two sample products:")
    for sample_key in SAMPLE_KEYS:
        print("\n---", sample_key, "---")
        plot_product(s3, BUCKET, sample_key)


## Citation and next steps

Please cite PDS datasets using the guidance on the [Geosciences Node FAQ](https://pds-geosciences.wustl.edu/faq.html). For large-scale analysis, combine anonymous S3 access with tools such as [STAC catalogs](https://stacspec.org/), [xarray](https://xarray.dev/), or [Dask](https://www.dask.org/) over `s3fs` once cloud-native formats (for example COG or Zarr) are published.

**Useful links**

- [PDS Geosciences Node](https://pds-geosciences.wustl.edu/)
- [PDS standards documentation](https://pds.nasa.gov/datastandards/documentation/)
- [AWS Open Data Registry](https://registry.opendata.aws/) (search for `pds-geosciences` when the entry merges)


## Community challenge

**Question:** *How does surface texture (roughness at 10–100 m scales) differ between ancient Noachian highlands and younger Amazonian volcanic plains on Mars, using orbital image products you stream from S3 without downloading the full archive?*

**Why it matters:** Geomorphological province boundaries drive landing-site safety, resource mapping, and climate history studies. Cloud access to PDS orbital data makes regional comparisons feasible on a laptop if you only pull the tiles you need.

**Suggested approach**

1. Discover MRO / Odyssey / MEX image or DEM products under `s3://pds-geosciences/` (once populated) with the browse helpers above.
2. Build a small AOI (area of interest) over one highland region and one plain region.
3. Compute a texture metric (for example variance of Laplacian, or GLCM contrast) on co-registered subsets.
4. Share a notebook that reads **only** the bytes required via range requests, and document citation metadata for each PDS product ID.

We would love to see your notebook linked from a GitHub issue or pull request in [this examples repository](https://github.com/fossettlab/open-data-examples).
